# Coerção de tipos strict e lax

Como vimos antes, o Pydantic faz coerção de tipos no modo **lax**.

In [1]:
from pydantic import BaseModel, ConfigDict, ValidationError

In [2]:
class Model(BaseModel):
    field_1: str
    field_2: float
    field_3: list
    field_4: tuple

Vejamos como a coerção lax lida com estes dados:

In [3]:
try:
    Model(field_1=100, field_2=1, field_3=(1, 2, 3), field_4=[1, 2, 3])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field_1
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


Com a coerção lax, o Pydantic aceitou converter tupla em lista, lista em tupla e inteiro em float. Só reclamou do inteiro em `str`.

Corrigindo só esse campo, vejamos o resultado:

In [4]:
print(f'Model(field_1="abc", field_2=1, field_3=(1, 2, 3), field_4=[1, 2, 3]): {Model(field_1="abc", field_2=1, field_3=(1, 2, 3), field_4=[1, 2, 3])!r}')

Model(field_1="abc", field_2=1, field_3=(1, 2, 3), field_4=[1, 2, 3]): Model(field_1='abc', field_2=1.0, field_3=[1, 2, 3], field_4=(1, 2, 3))


A lista e a tupla foram convertidas sem problema, assim como o inteiro para float.

Talvez não seja o comportamento desejado. Podemos escrever validadores customizados se precisarmos de validação de tipo especial
além do que o Pydantic oferece. Também podemos definir lax ou strict para o modelo inteiro (ou, como veremos depois, campo a campo).

Por ora, vamos colocar o modelo inteiro em modo **strict**.

Isso também é feito via `model_config`.

In [5]:
class Model(BaseModel):
    model_config = ConfigDict(strict=True)  # o padrão é False

    field_1: str
    field_2: float
    field_3: list
    field_4: tuple

E repetindo o exemplo:

In [6]:
try:
    Model(field_1=100, field_2=1, field_3=(1, 2, 3), field_4=[1, 2, 3])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 3 validation errors for Model
field_1
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
field_3
  Input should be a valid list [type=list_type, input_value=(1, 2, 3), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
field_4
  Input should be a valid tuple [type=tuple_type, input_value=[1, 2, 3], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/tuple_type


As exceções de validação mudaram.

> 📝 **Nota de revisão (Pydantic 2.5 → 2.13):** o curso foi gravado com o Pydantic 2.5 e afirmava que, em strict, converter **lista em
> tupla** falhava mas **tupla em lista** ainda era aceito. No 2.13 (saída acima) **as duas** falham: em strict, `list` só aceita `list`
> e `tuple` só aceita `tuple` (para dados Python). Já `int → float` continua aceito em strict — e é por isso que `field_2` não aparece
> nos erros.

Ainda assim, vale o alerta do autor: não assuma que **nenhuma** coerção acontece em strict. Algumas continuam acontecendo (como
`int → float`), menos que no lax, mas acontecem.

Para saber exatamente o que lax e strict significam, consulte a documentação [aqui](https://docs.pydantic.dev/latest/concepts/conversion_table/).

Procure o tipo `list` e veja se converter tupla em lista é suportado em strict. E em lax?

Faça o mesmo para tuplas: veja o que a documentação diz para lax e strict.

Até agora vimos coerção de tipos de dados **Python**.

Mas, ao desserializar **JSON**, o comportamento é um pouco diferente.

JSON é bem simples, não tem tantos tipos quanto o Python.

Basicamente, JSON entende estes tipos:
- **boolean**: `true` ou `false` (repare nas minúsculas)
- **number** (float ou inteiro, tanto faz)
- **string** (entre aspas duplas)
- **null** (representado por `null`, sem aspas)
- **array** (colchetes, valores separados por vírgula de qualquer tipo JSON, inclusive arrays e objects)
- **object** (basicamente um dicionário, entre chaves, pares `chave: valor`; chaves são strings e valores podem ser de qualquer tipo JSON,
  inclusive outro object)

Por exemplo:

In [7]:
json_data = '''
{
    "field_1": true,
    "field_2": 10.5,
    "field_3": 10,
    "field_4": null,
    "field_5": [1, 2, 3],
    "field_6": {
        "a": 1,
        "b": 2,
        "c": [3, 4, 5]
    },
    "field_7": [
        [1, 0, 0],
        [0, 1, 0],
        [0, 0, 1]
    ]
}
'''

O Python desserializa JSON nativamente (de JSON para dicionário) com a função `loads()` do módulo `json`.

In [8]:
import json
from pprint import pprint

data = json.loads(json_data)
pprint(data)

{'field_1': True,
 'field_2': 10.5,
 'field_3': 10,
 'field_4': None,
 'field_5': [1, 2, 3],
 'field_6': {'a': 1, 'b': 2, 'c': [3, 4, 5]},
 'field_7': [[1, 0, 0], [0, 1, 0], [0, 0, 1]]}


O Python precisou fazer suposições sobre os tipos — ele também faz a sua própria coerção.

Mas talvez não seja exatamente a que queremos.

Por exemplo, todos os arrays viraram listas — talvez quiséssemos tuplas.

O valor `10` virou `int` — talvez quiséssemos float.

E os objects viraram dicionários Python — talvez quiséssemos um modelo Pydantic no lugar de um dicionário (veremos modelos aninhados
mais adiante no curso).

In [9]:
print(f"type(data['field_3']): {type(data['field_3'])!r}")

type(data['field_3']): <class 'int'>


Então podemos transformar isso num modelo Pydantic e obter os tipos que queremos.

Lembre que, com tipos Python em strict, a coerção de lista para tupla falhou.

Porém, ao desserializar **JSON** com o Pydantic, não é o caso. Na documentação acima você verá que arrays JSON podem virar listas, tuplas,
sets e mais alguns tipos — com regras um pouco mais rígidas em strict.

Vamos criar um modelo para este JSON:

In [10]:
json_data = '''
{
    "field_1": true,
    "field_2": 10,
    "field_3": 1,
    "field_4": null,
    "field_5": [1, 2, 3],
    "field_6": ["a", "b", "c"],
    "field_7": {"a": 1, "b": 2}
}
'''

Queremos os seguintes tipos:
- `field_1`: bool
- `field_2`: float
- `field_3`: int
- `field_4`: string anulável
- `field_5`: tupla de inteiros
- `field_6`: set de strings
- `field_7`: dicionário

In [11]:
class Model(BaseModel):
    field_1: bool
    field_2: float
    field_3: int
    field_4: str | None
    field_5: tuple[int, ...]
    field_6: set[str]
    field_7: dict

Podemos até usar o modo strict, se preferirmos:

In [12]:
class Model(BaseModel):
    model_config = ConfigDict(strict=True)

    field_1: bool
    field_2: float
    field_3: int
    field_4: str | None
    field_5: tuple[int, ...]
    field_6: set[str]
    field_7: dict

In [13]:
data = Model.model_validate_json(json_data)

Agora não só desserializamos o JSON, como temos os tipos corretos e os dados validados (por exemplo: só aceitamos uma tupla de inteiros,
ou de valores conversíveis para inteiro).

In [14]:
print(f"data: {data!r}")

data: Model(field_1=True, field_2=10.0, field_3=1, field_4=None, field_5=(1, 2, 3), field_6={'a', 'c', 'b'}, field_7={'a': 1, 'b': 2})


In [15]:
print(f"type(data.field_5): {type(data.field_5)!r}")

type(data.field_5): <class 'tuple'>


In [16]:
print(f"type(data.field_6): {type(data.field_6)!r}")

type(data.field_6): <class 'set'>


In [17]:
print(f"type(data.field_7): {type(data.field_7)!r}")

type(data.field_7): <class 'dict'>


Se tentarmos carregar um JSON que não segue o schema — por exemplo, um float na tupla ou um inteiro no set:

In [18]:
json_data = '''
{
    "field_1": true,
    "field_2": 10,
    "field_3": 1,
    "field_4": null,
    "field_5": [1, 2, 3.5],
    "field_6": ["a", "b", 100],
    "field_7": {"a": 1, "b": 2}
}
'''

try:
    Model.model_validate_json(json_data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
field_5.2
  Input should be a valid integer [type=int_type, input_value=3.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type
field_6.2
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
